# Bronze - Silver | CineData Analytics

**Objetivo:** limpar, tipar, traduzir colunas para português e deduplicar os dados da Bronze. **A Bronze nunca é alterada.**

| # | Tabela Silver | Origem (Bronze) | Principais regras |
|---|---|---|---|
| 1 | `silver.tb_cotacao_dolar` | `tb_cotacao_dolar` | série diária contínua + *forward fill* |
| 2 | `silver.tb_info_filmes` | `tb_movies_info` | dedup por ingestão, status traduzido, data multi-formato |
| 3 | `silver.tb_financeiro_filmes` | `tb_movies_financials` | limpeza monetária, BRL, lucro e margem |
| 4 | `silver.tb_metricas_engajamento` | `tb_movies_metrics` | conversão segura, limites de negócio (nota 0–10) |
| 5 | `silver.tb_avaliacoes_usuarios` | `tb_movies_reviews` | dedup, nota 0–10, "Sem comentário" |
| 6 | `silver.tb_generos` | `tb_credits_and_tags.genres` | split/explode + domínio de gêneros |
| 7 | `silver.tb_pessoas_empresas` | `tb_credits_and_tags` (4 colunas) | dimensão unificada Ator/Diretor/Roteirista/Produtora |

> A cotação (1) é processada **primeiro** porque o financeiro (3) depende dela para converter USD → BRL.

**Princípio de robustez:** nenhuma conversão de tipo usa `CAST` "puro". Usamos `try_cast` / `try_to_timestamp`, que devolvem `NULL`
em vez de lançar erro — assim, dados corrompidos (Column Shift, textos fora de contexto) nunca derrubam o pipeline,
inclusive em compute *serverless* (onde o modo ANSI vem ligado).

In [0]:
import re
from datetime import date
from functools import reduce
from pyspark.sql import functions as F, Window, DataFrame

dbutils.widgets.text("catalog", "workspace", "1. Catálogo (vazio = padrão da sessão)")
CATALOG = dbutils.widgets.get("catalog").strip()
if CATALOG:
    spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

try:
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
except Exception as e:
    print("Aviso: não foi possível ajustar timeParserPolicy:", e)

TOKENS_AUSENTES = ["unknown", "não informado", "nao informado", "n/a", "na", "null", "none", "nan",
                   "-", "--", "?", "undefined", "desconhecido", "sem informação", "sem informacao"]

def limpa_txt(c):
    """Remove espaços (inclusive NBSP) das pontas; string vazia vira NULL."""
    s = F.regexp_replace(c.cast("string"), r"^[\s\u00A0]+|[\s\u00A0]+$", "")
    return F.when(s == "", F.lit(None)).otherwise(s)

def limpa_ausente(c):
    """Converte tokens como 'Unknown' / 'Não Informado' em NULL (antes de qualquer conversão de tipo)."""
    s = limpa_txt(c)
    return F.when(F.lower(s).isin(TOKENS_AUSENTES), F.lit(None)).otherwise(s)

def norm_id(nome_coluna: str):
    """Chave natural do filme como STRING limpa. '123.0' -> '123'. Valores que não parecem ID (texto deslocado) -> NULL."""
    s = F.regexp_replace(limpa_txt(F.col(nome_coluna)), r"\.0+$", "")
    return F.when(s.rlike(r"^[A-Za-z0-9_\-]+$"), s)

def limpa_decimal(c):
    """
    Prepara texto para conversão em DECIMAL/DOUBLE (popularidade, notas):
      - qualquer valor com LETRAS é texto fora de contexto (Column Shift) => NULL
      - remove pontuação/símbolos estranhos, troca vírgula decimal por ponto
      - se sobrarem vários pontos (ex.: '1.234.56'), o ÚLTIMO é o decimal e os demais são separadores de milhar
    """
    t = limpa_txt(c)
    s = F.regexp_replace(t, r"[^0-9,.\-]", "")
    s = F.regexp_replace(s, ",", ".")
    s = F.regexp_replace(s, r"\.(?=.*\.)", "")      
    s = F.regexp_replace(s, r"\.$", "")             
    return F.when(t.rlike(r"\p{L}") | t.isNull() | (s == ""), F.lit(None)).otherwise(s)

def limpa_moeda(c):
    """
    Prepara texto monetário: remove US$, R$, $, USD, BRL, €, £ e separadores de milhar.
      '$1,234,567.89' -> 1234567.89 | '1.234.567,89' -> 1234567.89 | '1.000.000' -> 1000000 | '2.37E8' -> 2.37E8
    Regra do separador: se terminar em [.,] + 1 ou 2 dígitos, é decimal; caso contrário, é milhar.
    """
    t = limpa_txt(F.regexp_replace(c.cast("string"), r"(?i)(US\$|R\$|USD|BRL|\$|€|£)", ""))
    cientifica = t.rlike(r"^-?\d+(\.\d+)?[eE][+-]?\d+$")
    s = F.regexp_replace(t, r"[^0-9,.\-]", "")
    tem_decimal = s.rlike(r"[.,]\d{1,2}$")
    parte_int = F.regexp_replace(F.regexp_extract(s, r"^(.*)[.,]\d{1,2}$", 1), r"[.,]", "")
    parte_dec = F.regexp_extract(s, r"[.,](\d{1,2})$", 1)
    numero = F.when(tem_decimal, F.concat(parte_int, F.lit("."), parte_dec)).otherwise(F.regexp_replace(s, r"[.,]", ""))
    return (F.when(cientifica, t)
             .when(t.isNull() | t.rlike(r"\p{L}") | (s == ""), F.lit(None))
             .otherwise(numero))

def limpa_inteiro(c):
    """
    Contagens (votos): aceita '1234', '1,234', '1.234' (milhar) e '1234.0'. Qualquer outra coisa
    (texto, decimal de verdade como '7.8') é dado deslocado => NULL.
    """
    t = limpa_txt(c)
    s = F.regexp_replace(t, r"[^0-9,.\-]", "")
    return (F.when(t.isNull() | t.rlike(r"\p{L}"), F.lit(None))
             .when(s.rlike(r"^-?\d{1,3}([.,]\d{3})+$"), F.regexp_replace(s, r"[.,]", ""))
             .when(s.rlike(r"^-?\d+[.,]0+$"), F.regexp_replace(s, r"[.,]0+$", ""))
             .when(s.rlike(r"^-?\d+$"), s)
             .otherwise(F.lit(None)))

def add_num(df, origem: str, destino: str, tipo: str, limpeza):
    """Aplica a limpeza e converte com try_cast (NULL em caso de falha, nunca erro)."""
    return (df.withColumn("_t", limpeza(F.col(origem)))
              .withColumn(destino, F.expr(f"try_cast(_t AS {tipo})"))
              .drop("_t"))

def add_int(df, origem: str, destino: str):
    """Inteiro (INT) >= 0. Votos/índices negativos ou fora da faixa de INT são inválidos => NULL."""
    df = add_num(df, origem, "_bi", "bigint", limpa_inteiro)
    return (df.withColumn(destino, F.when((F.col("_bi") >= 0) & (F.col("_bi") <= 2147483647), F.col("_bi").cast("int")))
              .drop("_bi"))

FORMATOS_DATA = [
    "yyyy-MM-dd", "yyyy/MM/dd", "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy", "dd.MM.yyyy",
    "d/M/yyyy", "M/d/yyyy", "yyyy-M-d", "yyyyMMdd",
    "MMMM d, yyyy", "MMM d, yyyy", "d MMMM yyyy", "d MMM yyyy",
]

def parse_data(df, origem: str, destino: str):
    """Testa cada formato com try_to_timestamp e usa o primeiro que funcionar. Se nenhum funcionar => NULL."""
    df = df.withColumn("_dt_str", limpa_txt(F.col(origem)))
    tentativas = [F.expr("try_to_timestamp(_dt_str, '" + fmt.replace("'", "\\'") + "')") for fmt in FORMATOS_DATA]
    return df.withColumn(destino, F.to_date(F.coalesce(*tentativas))).drop("_dt_str")

def dedup_recente(df, chave: str = "id_filme"):
    """
    Mantém 1 registro por chave: o de MAIOR ingestion_datetime (versão mais recente).
    Desempate (mesma ingestão): registro mais completo (mais campos preenchidos).
    """
    cols = [c for c in df.columns if c not in (chave, "id", "ingestion_datetime")]
    completude = reduce(lambda a, b: a + b, [F.when(limpa_txt(F.col(c)).isNotNull(), 1).otherwise(0) for c in cols])
    hash_conteudo = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]), 256)
    w = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc(), completude.desc(), hash_conteudo)
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def salvar_silver(df, tabela: str):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"✔ {tabela:<36} {spark.table(tabela).count():>10} linhas")

## 1) `silver.tb_cotacao_dolar` — série temporal contínua com *Forward Fill*

1. Converte `dataHoraCotacao` para timestamp e reduz a **uma cotação por dia** (a mais tardia do dia = fechamento PTAX).
2. Cria um **calendário completo** (todos os dias, inclusive finais de semana/feriados) entre a 1ª cotação e hoje.
3. **Forward fill:** dias sem cotação recebem o valor do último dia útil disponível (`last(..., ignorenulls=True)` numa janela acumulada).
4. A coluna `cotacao_preenchida` sinaliza quais dias foram preenchidos (transparência para auditoria).

In [0]:
cot = spark.table("bronze.tb_cotacao_dolar")

cot = (
    cot
    .withColumn("_ts", F.coalesce(
        F.expr("try_to_timestamp(trim(dataHoraCotacao), 'yyyy-MM-dd HH:mm:ss.SSSSSS')"),
        F.expr("try_to_timestamp(trim(dataHoraCotacao), 'yyyy-MM-dd HH:mm:ss.SSS')"),
        F.expr("try_to_timestamp(trim(dataHoraCotacao), 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(trim(dataHoraCotacao), 'yyyy-MM-dd')")))
    .withColumn("cotacao_compra", F.expr("try_cast(cotacaoCompra AS DOUBLE)"))
    .filter(F.col("_ts").isNotNull() & (F.col("cotacao_compra") > 0))   
    .withColumn("data_cotacao", F.to_date("_ts"))
)

w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("_ts").desc(), F.col("ingestion_datetime").desc())
cot_diaria = (cot.withColumn("_rn", F.row_number().over(w_dia)).filter("_rn = 1")
                 .select("data_cotacao", "cotacao_compra"))

limites = cot_diaria.agg(F.min("data_cotacao").alias("mn"), F.max("data_cotacao").alias("mx")).first()
if limites["mn"] is None:
    raise RuntimeError("bronze.tb_cotacao_dolar não possui cotações válidas.")
fim_calendario = max(limites["mx"], date.today())
calendario = spark.sql(
    f"SELECT explode(sequence(DATE'{limites['mn']}', DATE'{fim_calendario}', INTERVAL 1 DAY)) AS data_cotacao")

w_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
silver_cotacao = (
    calendario.join(cot_diaria, "data_cotacao", "left")
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())          
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_ffill))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
)

salvar_silver(silver_cotacao, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✔ silver.tb_cotacao_dolar                       6 linhas


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-14,5.169,false
2026-09-15,5.1484,false
2026-09-16,5.152,false
2026-09-17,5.1515,false
2026-09-18,5.1569,false
2026-09-19,5.1569,true


## 2) `silver.tb_info_filmes`

* **Deduplicação:** 1 linha por `id_filme`, mantendo a versão de maior `ingestion_datetime` (por isso a Bronze em *append* não é problema).
* **Status:** normaliza (minúsculas, sem acento, sem hífens/ruído: `"Post-Production"`, `" POST PRODUCTION "` → `postproduction`) e só então traduz.
  O que não casar com nenhum status conhecido vira **"Não Informado"**.
* **Data de lançamento:** testa vários formatos (`parse_data`); só vira `NULL` quando nenhum formato funciona.
* `ano_lancamento` é derivado de `data_lancamento`.

In [0]:
def chave_status(c):
    s = F.translate(F.lower(limpa_txt(c)), "áàâãäéèêëíìîïóòôõöúùûüçñ", "aaaaaeeeeiiiiooooouuuucn")
    return F.regexp_replace(s, r"[^a-z]", "")

MAPA_STATUS = {
    "released": "Lançado",       "lancado": "Lançado",
    "postproduction": "Pós-Produção",  "posproducao": "Pós-Produção",
    "inproduction": "Em Produção",     "emproducao": "Em Produção",
    "planned": "Planejado",      "planejado": "Planejado",
    "rumored": "Rumores",        "rumoured": "Rumores",   "rumores": "Rumores",
    "canceled": "Cancelado",     "cancelled": "Cancelado", "cancelado": "Cancelado",
}

def traduz_status(c):
    k = chave_status(c)
    expr = None
    for chave, pt in MAPA_STATUS.items():
        expr = F.when(k == chave, pt) if expr is None else expr.when(k == chave, pt)
    return expr.otherwise("Não Informado")      

b_info = (spark.table("bronze.tb_movies_info")
          .withColumn("id_filme", norm_id("id"))
          .filter(F.col("id_filme").isNotNull()))
b_info = dedup_recente(b_info)                                    

silver_info = (
    b_info
    .transform(lambda d: parse_data(d, "release_date", "data_lancamento"))
    .transform(lambda d: add_num(d, "runtime", "_dur", "double", limpa_decimal))
    .select(
        "id_filme",
        limpa_txt(F.col("title")).alias("titulo"),
        limpa_txt(F.col("original_title")).alias("titulo_original"),
        F.col("data_lancamento"),
        F.year("data_lancamento").alias("ano_lancamento"),                                   
        F.when((F.col("_dur") > 0) & (F.col("_dur") <= 2147483647), F.col("_dur").cast("int")).alias("duracao_minutos"),
        limpa_txt(F.col("original_language")).alias("idioma_original"),
        traduz_status(F.col("status")).alias("status_filme"),
        limpa_txt(F.col("overview")).alias("sinopse"),
        limpa_txt(F.col("tagline")).alias("frase_divulgacao"),
    )
)
salvar_silver(silver_info, "silver.tb_info_filmes")
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))
display(spark.table("silver.tb_info_filmes").limit(10))

✔ silver.tb_info_filmes                     97611 linhas


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null
1000014,On va manquer !,On va manquer !,2018-05-15,2018,null,fr,Lançado,null,null
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,null,es,Lançado,null,null
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null
1000058,Homecoming,Le retour,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null
1000073,A Chance To Win,Pour l'honneur,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null


## 3) `silver.tb_financeiro_filmes`

* Tokens de ausência (`Unknown`, `Não Informado`…) → `NULL` **antes** da conversão.
* Remove símbolos de moeda e separadores de milhar; converte para `DECIMAL(18,2)`.
* Valores **zerados ou negativos** = ausentes (`NULL`) — orçamento/receita 0 na base TMDB significa "não informado", não "custou zero".
* **Cotação aplicada:** a API só traz os últimos dias, então usamos **a cotação mais recente disponível** para converter todos os valores (premissa documentada).
* **Lucro** = receita − orçamento, calculado **somente quando os dois valores existem** (senão `NULL`; não inventamos zero).
  **Margem %** = lucro / receita × 100, com guarda contra divisão por zero.

In [0]:
ultima = spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()
TAXA_USD_BRL, DATA_TAXA = float(ultima["cotacao_compra"]), ultima["data_cotacao"]
print(f"Taxa USD->BRL aplicada: {TAXA_USD_BRL} (cotação de {DATA_TAXA})")

b_fin = (spark.table("bronze.tb_movies_financials")
         .withColumn("id_filme", norm_id("id"))
         .filter(F.col("id_filme").isNotNull()))
b_fin = dedup_recente(b_fin)

fin = b_fin
for origem, destino in [("budget", "orcamento_usd"), ("revenue", "receita_usd")]:
    fin = add_num(fin, origem, "_v", "decimal(18,2)", lambda c: limpa_moeda(limpa_ausente(c)))
    fin = fin.withColumn(destino, F.when(F.col("_v") > 0, F.col("_v"))).drop("_v")

taxa_sql = f"CAST('{TAXA_USD_BRL}' AS DECIMAL(18,6))"
silver_fin = (
    fin
    .withColumn("orcamento_brl", F.expr(f"try_cast(orcamento_usd * {taxa_sql} AS DECIMAL(18,2))"))
    .withColumn("receita_brl",   F.expr(f"try_cast(receita_usd * {taxa_sql} AS DECIMAL(18,2))"))
    .withColumn("lucro_usd", F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
                                    F.expr("try_cast(receita_usd - orcamento_usd AS DECIMAL(18,2))")))
    .withColumn("lucro_brl", F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
                                    F.expr("try_cast(receita_brl - orcamento_brl AS DECIMAL(18,2))")))
    .withColumn("margem_lucro_pct", F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
                                           F.expr("try_cast(round(lucro_usd / receita_usd * 100, 2) AS DECIMAL(18,2))")))
    .withColumn("cotacao_usd_brl_aplicada", F.lit(TAXA_USD_BRL))
    .withColumn("data_cotacao_aplicada", F.lit(DATA_TAXA))
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_pct", "cotacao_usd_brl_aplicada", "data_cotacao_aplicada")
)
salvar_silver(silver_fin, "silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(10))

Taxa USD->BRL aplicada: 5.1569 (cotação de 2026-09-19)
✔ silver.tb_financeiro_filmes               99006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct,cotacao_usd_brl_aplicada,data_cotacao_aplicada
1000004,null,null,null,null,null,null,null,5.1569,2026-09-19
1000005,null,null,null,null,null,null,null,5.1569,2026-09-19
1000007,null,null,null,null,null,null,null,5.1569,2026-09-19
1000011,null,null,null,null,null,null,null,5.1569,2026-09-19
1000014,null,null,null,null,null,null,null,5.1569,2026-09-19
1000030,null,null,null,null,null,null,null,5.1569,2026-09-19
1000054,null,null,null,null,null,null,null,5.1569,2026-09-19
1000058,4700000.00,null,24237430.00,null,null,null,null,5.1569,2026-09-19
1000059,null,null,null,null,null,null,null,5.1569,2026-09-19
1000073,6000000.00,null,30941400.00,null,null,null,null,5.1569,2026-09-19


## 4) `silver.tb_metricas_engajamento`

* **Popularidade:** limpeza de pontuação/separador decimal (`"84,5"`, `"1.234,56"`, `"84.5!"`) *antes* do `try_cast`, para não virar `NULL` silenciosamente.
* **Column Shift:** qualquer texto nas colunas numéricas (ex.: `"Drama"`) → `NULL` via `try_cast` (o pipeline não quebra).
* **Regras de negócio:** notas TMDB e IMDb fora de **0–10** (inclui erro de escala, ex.: `78` em vez de `7.8`) → `NULL`;
  votos e popularidade **negativos** → `NULL`; votos precisam ser **inteiros**.

In [0]:
b_met = (spark.table("bronze.tb_movies_metrics")
         .withColumn("id_filme", norm_id("id"))
         .filter(F.col("id_filme").isNotNull()))
b_met = dedup_recente(b_met)

met = b_met
met = add_num(met, "popularity",    "_pop",  "double", limpa_decimal)
met = add_num(met, "vote_average",  "_nt",   "double", limpa_decimal)
met = add_num(met, "averageRating", "_ni",   "double", limpa_decimal)
met = add_int(met, "vote_count", "_vt")
met = add_int(met, "numVotes",   "_vi")

def nota_valida(c):
    """Nota válida = entre 0 e 10 (inclusive). Fora disso (ex.: escala x10) => NULL."""
    return F.when((F.col(c) >= 0) & (F.col(c) <= 10), F.col(c))

silver_met = met.select(
    "id_filme",
    F.when(F.col("_pop") >= 0, F.col("_pop")).alias("popularidade"),          
    nota_valida("_nt").alias("nota_media_tmdb"),
    F.col("_vt").alias("qtd_votos_tmdb"),                                     
    nota_valida("_ni").alias("nota_media_imdb"),
    F.col("_vi").alias("qtd_votos_imdb"),
)
salvar_silver(silver_met, "silver.tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").limit(10))

✔ silver.tb_metricas_engajamento            95115 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236
1000081,102.802,5.142,134,5.1,1841
1000092,4.797,8.5,1,6.8,130
1000094,2.645,6.9,28,6.5,null
1000096,0.71,10.0,1,6.3,695
1000099,0.6,0.0,0,8.6,33
1000108,3.231,0.0,0,6.5,157
1000110,1.38,0.0,0,6.9,263


## 5) `silver.tb_avaliacoes_usuarios`

* Nota do usuário fora de **0–10** → `NULL`.
* Comentário vazio, nulo ou só com espaços → **"Sem comentário"**.
* Remove duplicatas integrais (mesmo filme + usuário + nota + comentário).

In [0]:
rev = (spark.table("bronze.tb_movies_reviews")
       .withColumn("id_filme", norm_id("id"))
       .filter(F.col("id_filme").isNotNull()))
rev = add_num(rev, "nota", "_nota", "double", limpa_decimal)

silver_rev = (
    rev.select(
        "id_filme",
        limpa_txt(F.col("nome")).alias("nome_usuario"),
        F.when((F.col("_nota") >= 0) & (F.col("_nota") <= 10), F.col("_nota")).alias("nota_usuario"),   
        F.coalesce(limpa_txt(F.col("comentario")), F.lit("Sem comentário")).alias("comentario_usuario"),   
    )
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)
salvar_silver(silver_rev, "silver.tb_avaliacoes_usuarios")
display(spark.table("silver.tb_avaliacoes_usuarios").limit(10))

✔ silver.tb_avaliacoes_usuarios             32412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


## 6) `silver.tb_generos`

* `credits_and_tags` é deduplicada por filme (versão mais recente) antes do *explode*.
* **Separadores inconsistentes:** `;` é trocado por `,` **antes** do `split`.
* **Resíduos (Column Shift / separadores extras):** cada token é validado contra o **domínio de gêneros** (lista de gêneros TMDB/IMDb).
  Tokens vazios, numéricos, descritivos (sinopses) ou nomes de pessoas deslocados **não pertencem ao domínio** e são descartados.
  No fim, o notebook exibe os tokens rejeitados mais frequentes — se algum gênero legítimo aparecer ali, basta incluí-lo em `GENEROS_DOMINIO`.
* Aliases (`Sci-Fi`) são padronizados para o nome canônico (`Science Fiction`).

In [0]:
b_cred = (spark.table("bronze.tb_credits_and_tags")
          .withColumn("id_filme", norm_id("id"))
          .filter(F.col("id_filme").isNotNull()))
b_cred = dedup_recente(b_cred)

def normaliza_chave(txt: str) -> str:
    """Mesma normalização em Python e em Spark: minúsculas, hífen/underscore/espaços múltiplos -> 1 espaço."""
    return re.sub(r"[\s\-_]+", " ", txt.strip().lower())

def chave_spark(c):
    return F.lower(F.regexp_replace(limpa_txt(c), r"[\s\-_]+", " "))

GENEROS_DOMINIO = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family", "Fantasy",
                   "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller", "War",
                   "Western", "Biography", "Musical", "Sport", "Film Noir", "News", "Reality TV", "Talk Show",
                   "Game Show", "Short", "Adult"]
ALIASES = {"Sci-Fi": "Science Fiction", "SciFi": "Science Fiction", "Sci Fi": "Science Fiction"}

dominio = {normaliza_chave(g): g for g in GENEROS_DOMINIO}
dominio.update({normaliza_chave(a): g for a, g in ALIASES.items()})
df_dominio = spark.createDataFrame(list(dominio.items()), ["chave_genero", "genero"])

tokens = (
    b_cred
    .select("id_filme", F.explode(F.split(F.regexp_replace(F.col("genres"), r"[;|]", ","), ",")).alias("token_bruto"))
    .withColumn("token", limpa_txt(F.col("token_bruto")))
    .filter(F.col("token").isNotNull())                                   
    .withColumn("chave_genero", chave_spark(F.col("token")))
)

silver_generos = (tokens.join(F.broadcast(df_dominio), "chave_genero", "inner")   
                        .select("id_filme", "genero").distinct())
salvar_silver(silver_generos, "silver.tb_generos")

print("Tokens descartados mais frequentes (auditoria):")
display(tokens.join(F.broadcast(df_dominio), "chave_genero", "left_anti")
              .groupBy("token").count().orderBy(F.desc("count")).limit(30))
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

✔ silver.tb_generos                        141940 linhas
Tokens descartados mais frequentes (auditoria):


token,count
0.6,265
United States of America,28
1.4,26
[],25
English,19
0.0,17
Nenhum,9
N/A,9
Japan,7
France,7


genero,count
Drama,32615
Documentary,19234
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
TV Movie,4115


## 7) `silver.tb_pessoas_empresas` — dimensão unificada

`cast → 'Ator'`, `directors → 'Diretor'`, `writers → 'Roteirista'`, `production_companies → 'Produtora'`.

* `posexplode` guarda a **ordem de créditos** (`ordem_creditos`) — usada na Gold para escolher os *atores principais*.
* Padroniza capitalização (`initcap`), remove aspas/colchetes das pontas e **duplicatas** (mesmo filme + nome + tipo).
* **Filtro de resíduos:** descarta tokens sem nenhuma letra (números deslocados), muito longos (texto descritivo),
  tokens de ausência e palavras do domínio de gêneros/status que "vazaram" de outras colunas.

In [0]:
TIPOS_ENTIDADE = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

FORA_DO_DOMINIO = set(dominio.keys()) | set(TOKENS_AUSENTES) | set(MAPA_STATUS.keys()) | {"released", "post production", "in production"}

partes = []
for coluna, tipo in TIPOS_ENTIDADE.items():
    partes.append(
        b_cred.select(
            "id_filme",
            F.posexplode(F.split(F.regexp_replace(F.col(coluna), r"[;|]", ","), ",")).alias("ordem_creditos", "nome_bruto"),
        ).withColumn("tipo_entidade", F.lit(tipo))
    )
todas = reduce(DataFrame.unionByName, partes)

nome_limpo = limpa_txt(F.regexp_replace(F.regexp_replace(F.col("nome_bruto"), r"^[\[\]'\"\s\u00A0]+|[\[\]'\"\s\u00A0]+$", ""), r"\s+", " "))

silver_pessoas = (
    todas.withColumn("nome_entidade", F.regexp_replace(F.initcap(F.regexp_replace(nome_limpo, "-", "- ")), "- ", "-"))   
    .filter(
        F.col("nome_entidade").isNotNull()
        & F.col("nome_entidade").rlike(r"\p{L}")                                   
        & (F.length("nome_entidade") <= 80)                                        
        & (F.size(F.split("nome_entidade", " ")) <= 8)
        & ~chave_spark(F.col("nome_entidade")).isin(list(FORA_DO_DOMINIO))         
    )
    .groupBy("id_filme", "nome_entidade", "tipo_entidade").agg(F.min("ordem_creditos").alias("ordem_creditos"))
)
salvar_silver(silver_pessoas, "silver.tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())
display(spark.table("silver.tb_pessoas_empresas").orderBy("id_filme", "tipo_entidade", "ordem_creditos").limit(20))

✔ silver.tb_pessoas_empresas               889142 linhas


tipo_entidade,count
Ator,543545
Diretor,101426
Roteirista,125960
Produtora,118211


id_filme,nome_entidade,tipo_entidade,ordem_creditos
1000004,Izzy Jones,Ator,0
1000004,Erika Alexander,Ator,1
1000004,Aron Von Andrian,Ator,2
1000004,Steven Michael-O’hara,Ator,3
1000004,Tedroy Newell,Ator,4
1000004,Lola Atkins,Diretor,0
1000005,Aisha Brown,Ator,0
1000005,Mathieu Baer,Diretor,0
1000005,Just For Laughs Television,Produtora,0
1000007,Kyle Brownrigg,Ator,0


## 8) Conferência final da Silver
Esquema (tipos) e perfil de nulos de cada tabela — evidência de que a tipagem está correta e que valores inválidos viraram `NULL`.

In [0]:
for t in ["silver.tb_info_filmes", "silver.tb_financeiro_filmes", "silver.tb_metricas_engajamento",
          "silver.tb_avaliacoes_usuarios", "silver.tb_generos", "silver.tb_pessoas_empresas", "silver.tb_cotacao_dolar"]:
    df = spark.table(t)
    print(f"\n=== {t} ({df.count()} linhas) ===")
    df.printSchema()
    display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]))


=== silver.tb_info_filmes (97611 linhas) ===
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)



id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
0,0,0,2,2,10165,0,0,13232,74784



=== silver.tb_financeiro_filmes (99006 linhas) ===
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_pct: decimal(18,2) (nullable = true)
 |-- cotacao_usd_brl_aplicada: double (nullable = true)
 |-- data_cotacao_aplicada: date (nullable = true)



id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct,cotacao_usd_brl_aplicada,data_cotacao_aplicada
0,91244,95734,91244,95734,97536,97536,97536,0,0



=== silver.tb_metricas_engajamento (95115 linhas) ===
root
 |-- id_filme: string (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



...

id_filme,genero
0,0



=== silver.tb_pessoas_empresas (889142 linhas) ===
root
 |-- id_filme: string (nullable = true)
 |-- nome_entidade: string (nullable = true)
 |-- tipo_entidade: string (nullable = true)
 |-- ordem_creditos: integer (nullable = true)



id_filme,nome_entidade,tipo_entidade,ordem_creditos
0,0,0,0



=== silver.tb_cotacao_dolar (6 linhas) ===
root
 |-- data_cotacao: date (nullable = true)
 |-- cotacao_compra: double (nullable = true)
 |-- cotacao_preenchida: boolean (nullable = true)



data_cotacao,cotacao_compra,cotacao_preenchida
0,0,0
